In [ ]:
#!/usr/bin/env python3
"""
v5 G0 — two-component reproducibility, REPLOGLE K562 leg.  KAGGLE, CPU-only.
Engineer, 2026-07-19. Per Brian's ruling of 2026-07-19 (P-sweep is now a
mandatory control ranked alongside the shuffled null).

Parent brief : docs/TASK_v5_G0_two_component_reproducibility.md (as amended)
Prior leg    : docs/v5_G0_scbloodnl_pipeline_result.md  (pipeline validated there)

------------------------------------------------------------------------------
ENVIRONMENT (Kaggle, free tier, CPU)
  Add dataset: Replogle et al. 2022 processed Perturb-seq (Figshare+ 20029387),
  file `K562_essential_raw_singlecell_01.h5ad` (~9.9 GB).
  Needs: h5py, numpy, pandas, matplotlib. No GPU. No scanpy (we read the HDF5
  directly and never materialise a dense matrix).
  RAM: peak ~4 GB with default settings.

------------------------------------------------------------------------------
LOCKED DESIGN (pre-registered; do not edit after seeing output)

  perturbation p : TARGET GENE, with all guides for that gene POOLED.
                   Rationale, stated per Brian's request because this choice
                   moves the axis that now carries the gate:
                     1. VCC's task is gene-level. Comparability is the point.
                     2. Guide-level would push guide-specific off-target effects
                        into s_p, inflating the specific component with technical
                        rather than biological variance — an upward bias toward
                        MODELING FORK. Refusing it is the conservative call.
                     3. Depth is the scarcest resource in this dataset (see the
                        CEILING note below). Pooling guides preserves 2-3x more
                        cells per perturbation, and n is load-bearing.
                   Fixed across the entire sweep.
  control (ntc)  : 'non-targeting', split into its own independent A/B halves at
                   the same depth. Never shared across halves.
  n              : cells per HALF (halves disjoint => depth n consumes 2n cells).
                   Same convention as the scBloodNL leg, so the two are comparable.
  g              : recomputed inside each half AND inside each P-subsample.
  normalisation  : sum counts -> CPM -> log1p.
  gene filter    : detected in >= 1% of cells.
  seeds          : 3 real + 3 shuffled-label.

  DEPTH CEILING (measured, not assumed — verified from the CZI dataset card and
  re-verified from the file at runtime by stage 1):
    K562 essential has a median of >100 cells per perturbation. With n = cells
    per half, the reachable n is ~50-100. The locked decision bracket of
    1,000-3,000 cells/perturbation is therefore NOT REACHABLE on this dataset,
    by a factor of 10-60x. This leg can execute the P-sweep, the P=6 inflation
    check, and matched-P/matched-n cross-dataset comparison. It CANNOT evaluate
    the decision rule at the competition depth. Reported explicitly; see
    `ceiling_note` in the output JSON.

  P-SWEEP (mandatory control, per the 2026-07-19 ruling)
    P in {6, 25, 100, 500, max}, subsampled from the available targets, several
    draws per P, evaluated at MATCHED n.

    Two competing predictions are instrumented; Brian rules, this script does not.
      CHECK 1 (pre-registered, Brian): from
          s_p = ((P-1)/P)*d_p - (1/P)*sum_{q!=p} d_q
      at P=6, s_p ~= (5/6)d_p, so specific reproducibility should inherit total
      reproducibility. Subsampling to P=6 must reproduce the scBloodNL inflation.
      CHECK 2 (proposed, Engineer): correlation is SCALE-INVARIANT, so the
      (P-1)/P factor cannot move it, and the remaining term subtracts a quantity
      that is itself reproducible across halves. Prediction: specific
      reproducibility is ~P-INVARIANT, while frac_global falls as P rises.
      Simulation supports CHECK 2 — see src/v5g0_03_P_dependence_simulation_v1.py.

    Both are computed and reported. Nothing is self-authorised here.

USAGE
  Stage 1 (slow, once):  python v5g0_replogle_k562_kaggle_v1.py extract
  Stage 2 (fast, iterate): python v5g0_replogle_k562_kaggle_v1.py analyze
"""
import json
import os
import sys
import time

import h5py
import numpy as np
import pandas as pd

# ----------------------------------------------------------------- config --
H5AD = os.environ.get(
    "REPLOGLE_H5AD",
    "/kaggle/input/replogle-2022-k562-essential/K562_essential_raw_singlecell_01.h5ad")
OUT = os.environ.get("V5G0_OUT", "/kaggle/working")
os.makedirs(OUT, exist_ok=True)
ACCUM_NPZ = os.path.join(OUT, "v5g0_replogle_k562_accum_v1.npz")

CONTROL_LABELS = ("non-targeting", "non_targeting", "control", "NTC")
DEPTH_GRID = [10, 25, 50, 100, 250]        # cells per half; trimmed to what fits
P_GRID = [6, 25, 100, 500]                 # 'max' appended at runtime
N_SEED = 3
N_DRAW_P = 8                               # independent perturbation draws per P
MIN_DETECT_FRAC = 0.01
ROW_BLOCK = 20_000                         # cells per HDF5 read block

t0 = time.time()
def log(m): print(f"[{time.time()-t0:7.1f}s] {m}", flush=True)


# ============================================================== STAGE 1 =====
def read_obs_column(f, name):
    """Read an obs column, decoding AnnData categorical encoding if present."""
    node = f["obs"][name]
    if isinstance(node, h5py.Group):                      # categorical
        cats = node["categories"][:]
        codes = node["codes"][:]
        cats = np.array([c.decode() if isinstance(c, bytes) else str(c) for c in cats])
        return cats[codes]
    arr = node[:]
    if arr.dtype.kind in "SO":
        return np.array([a.decode() if isinstance(a, bytes) else str(a) for a in arr])
    return arr


def find_target_column(f):
    """Locate the perturbation-target column. Fail LOUDLY rather than guess."""
    keys = list(f["obs"].keys())
    for cand in ("gene", "gene_symbol", "target_gene", "perturbation", "gene_id"):
        if cand in keys:
            return cand
    raise KeyError(
        f"Could not find a perturbation-target column in obs. Available: {keys}. "
        "Set it explicitly before running — do not let this fall through to a guess.")


def extract():
    log(f"opening {H5AD}")
    with h5py.File(H5AD, "r") as f:
        log(f"obs columns: {list(f['obs'].keys())}")
        log(f"var columns: {list(f['var'].keys())}")

        tgt_col = find_target_column(f)
        log(f"using obs['{tgt_col}'] as the perturbation target")
        target = read_obs_column(f, tgt_col)

        # gene names
        vkeys = list(f["var"].keys())
        gname_key = next((k for k in ("gene_name", "gene_symbol", "_index", "index")
                          if k in vkeys), vkeys[0])
        gnode = f["var"][gname_key]
        genes = np.array([g.decode() if isinstance(g, bytes) else str(g)
                          for g in (gnode["categories"][:][gnode["codes"][:]]
                                    if isinstance(gnode, h5py.Group) else gnode[:])])

        # X: CSR on disk
        X = f["X"]
        if isinstance(X, h5py.Group):
            indptr = X["indptr"][:]
            data_ds, idx_ds = X["data"], X["indices"]
            n_cells = len(indptr) - 1
            n_genes = len(genes)
        else:
            raise TypeError("X is dense; this script expects CSR sparse AnnData.")
        log(f"{n_cells:,} cells x {n_genes:,} genes, {indptr[-1]:,} nonzeros")

        # ---- group cells by target gene (guides pooled) --------------------
        ctrl_mask = np.isin(target, CONTROL_LABELS)
        if not ctrl_mask.any():
            raise ValueError(
                f"No control cells found. Labels tried: {CONTROL_LABELS}. "
                f"Observed examples: {pd.unique(target)[:20]}")
        log(f"control cells ('{target[ctrl_mask][0]}'): {int(ctrl_mask.sum()):,}")

        labels = target.copy()
        labels[ctrl_mask] = "__CTRL__"
        uniq, inv = np.unique(labels, return_inverse=True)
        counts = np.bincount(inv)
        ctrl_gi = int(np.where(uniq == "__CTRL__")[0][0])

        pert_gi = np.array([i for i in range(len(uniq)) if i != ctrl_gi])
        log(f"targets (guides pooled): {len(pert_gi):,}")
        q = np.percentile(counts[pert_gi], [10, 25, 50, 75, 90])
        log(f"cells/target percentiles 10/25/50/75/90: {q.round(0)}")
        log(f"CEILING: median cells/target = {np.median(counts[pert_gi]):.0f} "
            f"-> max n per half ~ {np.median(counts[pert_gi])//2:.0f}")

        # ---- depth grid: keep grid points where >=80% of targets qualify ----
        depths = [d for d in DEPTH_GRID
                  if (counts[pert_gi] >= 2 * d).mean() >= 0.80 and counts[ctrl_gi] >= 2 * d]
        if not depths:
            raise ValueError("No depth grid point retains 80% of targets.")
        n_max = max(depths)
        log(f"depth grid (cells per half): {depths}")

        # keep only targets with enough cells at the deepest grid point, so the
        # perturbation SET is identical across depths and P-draws (matched design)
        keep = pert_gi[counts[pert_gi] >= 2 * n_max]
        log(f"targets retained at n={n_max}: {len(keep):,} of {len(pert_gi):,}")
        groups = np.concatenate([[ctrl_gi], keep])
        gpos = -np.ones(len(uniq), dtype=np.int64)
        gpos[groups] = np.arange(len(groups))
        N_GROUP = len(groups)
        N_DEPTH = len(depths)

        # ---- per-run bin assignment (prefix-sum trick, as in scBloodNL) -----
        RUNS = [("real", s) for s in range(N_SEED)] + [("shuffled", s) for s in range(N_SEED)]
        bounds = np.array(depths, dtype=np.int64)

        def build_bins(kind, seed):
            rng = np.random.default_rng(7000 * seed + (0 if kind == "real" else 31))
            lab = inv.copy()
            if kind == "shuffled":
                # permute target labels among ALL cells in the analysis set,
                # preserving cells-per-target exactly
                sel = np.where(gpos[inv] >= 0)[0]
                v = lab[sel].copy()
                rng.shuffle(v)
                lab[sel] = v
            out = np.full(n_cells, -1, dtype=np.int32)
            for gi in groups:
                sel = np.where(lab == gi)[0]
                perm = rng.permutation(sel.size)
                rank = np.empty(sel.size, dtype=np.int64)
                rank[perm] = np.arange(sel.size)
                k = rank < 2 * n_max
                r = rank[k]
                half = (r >= n_max).astype(np.int64)
                rr = r - half * n_max
                slot = np.minimum(np.searchsorted(bounds, rr, side="right"), N_DEPTH - 1)
                out[sel[k]] = (gpos[gi] * 2 + half) * N_DEPTH + slot
            return out

        bin_runs = np.stack([build_bins(k, s) for k, s in RUNS])
        BINS = N_GROUP * 2 * N_DEPTH
        log(f"{len(RUNS)} runs x {BINS:,} bins x {n_genes:,} genes "
            f"= {len(RUNS)*BINS*n_genes*4/1e9:.2f} GB float32")

        accum = np.zeros((len(RUNS), BINS, n_genes), dtype=np.float32)
        detect = np.zeros(n_genes, dtype=np.int64)

        # ---- stream cell blocks --------------------------------------------
        for start in range(0, n_cells, ROW_BLOCK):
            stop = min(start + ROW_BLOCK, n_cells)
            lo, hi = int(indptr[start]), int(indptr[stop])
            dat = data_ds[lo:hi].astype(np.float64)
            ind = idx_ds[lo:hi].astype(np.int64)
            rowlen = np.diff(indptr[start:stop + 1])
            rows = np.repeat(np.arange(start, stop), rowlen)

            detect += np.bincount(ind, minlength=n_genes)

            for ri in range(len(RUNS)):
                b = bin_runs[ri][rows]
                m = b >= 0
                if not m.any():
                    continue
                accum[ri] += np.bincount(
                    b[m] * n_genes + ind[m], weights=dat[m], minlength=BINS * n_genes
                ).reshape(BINS, n_genes).astype(np.float32)

            if (start // ROW_BLOCK) % 20 == 0:
                log(f"  cells {stop:,}/{n_cells:,} ({stop/n_cells:.1%})")

        np.savez(ACCUM_NPZ,
                 accum=accum, detect=detect, genes=genes,
                 group_names=uniq[groups], depths=np.array(depths),
                 n_max=n_max, n_cells_total=n_cells,
                 counts=counts[groups],
                 run_kind=np.array([k for k, _ in RUNS]),
                 run_seed=np.array([s for _, s in RUNS]),
                 median_cells_per_target=float(np.median(counts[pert_gi])),
                 n_targets_all=len(pert_gi))
        log(f"wrote {ACCUM_NPZ}")


# ============================================================== STAGE 2 =====
def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


def analyze():
    z = np.load(ACCUM_NPZ, allow_pickle=True)
    accum, detect, genes = z["accum"], z["detect"], z["genes"]
    group_names = list(z["group_names"])
    depths = list(z["depths"])
    run_kind, run_seed = list(z["run_kind"]), list(z["run_seed"])
    n_cells_total = int(z["n_cells_total"])
    N_DEPTH = len(depths)
    N_GROUP = len(group_names)
    n_genes = len(genes)

    gm = (detect / n_cells_total) >= MIN_DETECT_FRAC
    log(f"{gm.sum():,} / {n_genes:,} genes pass detection filter")

    CTRL = group_names.index("__CTRL__")
    pert_pos = [i for i in range(N_GROUP) if i != CTRL]
    P_max = len(pert_pos)
    p_grid = [p for p in P_GRID if p < P_max] + [P_max]
    log(f"P grid: {p_grid}  (P_max = {P_max})")

    def bid(gpos, half, slot):
        return (gpos * 2 + half) * N_DEPTH + slot

    rows = []
    for ri, (kind, seed) in enumerate(zip(run_kind, run_seed)):
        A = accum[ri]
        for slot in range(N_DEPTH):
            depth = int(depths[slot])
            # cumulative pseudobulk -> log-CPM, per group per half
            prof = np.empty((N_GROUP, 2, int(gm.sum())), dtype=np.float64)
            for gp in range(N_GROUP):
                for half in range(2):
                    # depth n is the prefix sum over depth bins 0..slot
                    s = A[bid(gp, half, 0):bid(gp, half, slot) + 1].sum(axis=0)
                    s = s[gm]
                    tot = s.sum()
                    prof[gp, half] = np.log1p(s / tot * 1e6) if tot > 0 else np.nan

            # deltas against the SAME half's control
            D = np.stack([prof[p, :, :] - prof[CTRL, :, :] for p in pert_pos])  # (P,2,G)

            rng = np.random.default_rng(999 * seed + slot)
            for P in p_grid:
                draws = 1 if P == P_max else N_DRAW_P
                for draw in range(draws):
                    sub = (np.arange(P_max) if P == P_max
                           else rng.choice(P_max, size=P, replace=False))
                    d = D[sub]                                  # (P,2,G)
                    g = d.mean(axis=0)                          # (2,G)
                    s = d - g[None, :, :]                       # (P,2,G)

                    r_tot = float(np.mean([corr(d[i, 0], d[i, 1]) for i in range(P)]))
                    r_spec = float(np.mean([corr(s[i, 0], s[i, 1]) for i in range(P)]))
                    r_glob = corr(g[0], g[1])

                    vt = float(np.mean([(d[i, h] ** 2).sum()
                                        for i in range(P) for h in range(2)]))
                    vg = float(np.mean([(g[h] ** 2).sum() for h in range(2)]))
                    vs = float(np.mean([(s[i, h] ** 2).sum()
                                        for i in range(P) for h in range(2)]))

                    rows.append(dict(
                        dataset="Replogle2022_K562_essential", run_kind=kind, seed=int(seed),
                        cells_per_pert=depth, P=P, draw=draw,
                        r_total=r_tot, r_global=r_glob, r_specific=r_spec,
                        inflation_ratio=r_spec / r_tot if r_tot else np.nan,
                        frac_global=vg / vt if vt else np.nan,
                        frac_specific=vs / vt if vt else np.nan,
                        n_genes=int(gm.sum())))
        log(f"  run {kind} seed {seed} done")

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(OUT, "v5g0_replogle_k562_reproducibility_v1.csv"), index=False)

    real = df[df.run_kind == "real"]
    shuf = df[df.run_kind == "shuffled"]
    n_deep = max(depths)

    summ = (real.groupby(["cells_per_pert", "P"])
            [["r_total", "r_global", "r_specific", "inflation_ratio",
              "frac_global", "frac_specific"]].mean().reset_index())
    summ.to_csv(os.path.join(OUT, "v5g0_replogle_k562_summary_v1.csv"), index=False)
    print("\n=== REAL: mean over seeds x draws ===")
    print(summ.round(4).to_string(index=False))

    # ---- HARD CHECK: does P=6 reproduce the scBloodNL inflation? ------------
    at_deep = real[real.cells_per_pert == n_deep]
    p6 = at_deep[at_deep.P == 6]
    pmax = at_deep[at_deep.P == max(p_grid)]
    infl_p6 = float(p6.inflation_ratio.mean())
    infl_pmax = float(pmax.inflation_ratio.mean())
    # CHECK 1 (pre-registered by Brian, 2026-07-19): specific/total must rise as P
    # falls and approach ~1 at P=6.
    check1_pass = bool(infl_p6 > infl_pmax and infl_p6 > 0.8)

    # CHECK 2 (proposed replacement, Engineer 2026-07-19; see
    # src/v5g0_03_P_dependence_simulation_v1.py). Correlation is scale-invariant, so
    # the (P-1)/P factor cannot move it; simulation says specific reproducibility is
    # ~P-invariant while the VARIANCE fraction is strongly P-dependent. Both arms are
    # instrumented so the numbers exist whichever way Brian rules. NOT self-authorised.
    fg_p6 = float(p6.frac_global.mean())
    fg_pmax = float(pmax.frac_global.mean())
    spec_p6 = float(p6.r_specific.mean())
    spec_pmax = float(pmax.r_specific.mean())
    check2_pass = bool(fg_p6 > fg_pmax and abs(spec_p6 - spec_pmax) < 0.05)

    report = {
        "task": "v5 G0 two-component reproducibility — Replogle K562 essential",
        "ruling": "Brian 2026-07-19: P-sweep mandatory; scBloodNL MODELING FORK does not count.",
        "design": {
            "perturbation_unit": "TARGET GENE, guides pooled (locked; see header for rationale)",
            "control": "non-targeting, split into independent A/B halves",
            "n_definition": "cells per half; depth n consumes 2n cells",
            "depths_cells_per_half": [int(d) for d in depths],
            "P_grid": [int(p) for p in p_grid],
            "seeds": int(len(set(run_seed))),
            "draws_per_P": N_DRAW_P,
        },
        "ceiling_note": {
            "median_cells_per_target": float(z["median_cells_per_target"]),
            "max_n_per_half_reached": int(n_deep),
            "decision_bracket": [1000, 3000],
            "bracket_reachable": False,
            "statement": (
                "K562 essential has a median of >100 cells per target. The locked "
                "1,000-3,000 cells/perturbation decision bracket is NOT reachable on this "
                "dataset, short by 10-60x. This leg tests the P-sweep, the P=6 inflation "
                "check, and matched-P/matched-n comparison. It does NOT evaluate the "
                "decision rule at competition depth."),
        },
        "controls": {
            "shuffled_specific": float(shuf.r_specific.mean()),
            "shuffled_global": float(shuf.r_global.mean()),
            "global_at_deepest_Pmax": float(pmax.r_global.mean()),
        },
        "check1_preregistered_P6_inflation": {
            "owner": "Brian, 2026-07-19 (pre-registered)",
            "inflation_ratio_at_P6": infl_p6,
            "inflation_ratio_at_Pmax": infl_pmax,
            "PASS": check1_pass,
            "note": ("PASS requires specific/total to rise as P falls and approach ~1 at P=6. "
                     "Engineer expects this to FAIL on the premise, not the pipeline — see "
                     "check2 and src/v5g0_03_P_dependence_simulation_v1.py."),
        },
        "check2_proposed_P_invariance": {
            "owner": "Engineer, 2026-07-19 (proposed, NOT self-authorised — awaiting ruling)",
            "frac_global_at_P6": fg_p6,
            "frac_global_at_Pmax": fg_pmax,
            "specific_corr_at_P6": spec_p6,
            "specific_corr_at_Pmax": spec_pmax,
            "PASS": check2_pass,
            "note": ("Correlation is scale-invariant, so the (P-1)/P factor cannot move it. "
                     "PASS requires frac_global to FALL as P rises while specific "
                     "reproducibility stays ~P-invariant (|delta| < 0.05). This inverts "
                     "check1's prediction and is equally falsifiable."),
        },
        "headline_at_deepest_n": {
            "n": int(n_deep),
            "P_max": int(max(p_grid)),
            "r_total": float(pmax.r_total.mean()),
            "r_global": float(pmax.r_global.mean()),
            "r_specific": float(pmax.r_specific.mean()),
            "frac_global": float(pmax.frac_global.mean()),
            "frac_specific": float(pmax.frac_specific.mean()),
        },
    }
    with open(os.path.join(OUT, "v5g0_replogle_k562_report_v1.json"), "w") as fh:
        json.dump(report, fh, indent=2)
    print("\n" + json.dumps({k: report[k] for k in
                             ("controls", "check1_preregistered_P6_inflation", "check2_proposed_P_invariance", "headline_at_deepest_n")},
                            indent=2))

    # ---- figures -----------------------------------------------------------
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for P in p_grid:
        sub = real[real.P == P].groupby("cells_per_pert").r_specific.mean()
        axes[0].plot(sub.index, sub.values, marker="o", label=f"P={P}")
    axes[0].set_xscale("log"); axes[0].set_xlabel("cells per perturbation (per half)")
    axes[0].set_ylabel("specific reproducibility"); axes[0].legend(fontsize=8)
    axes[0].axhline(0.3, ls=":", c="grey"); axes[0].axhline(0.6, ls=":", c="grey")
    axes[0].set_title("depth curve, by P")

    for comp, c in [("r_total", "#333"), ("r_global", "#1f77b4"), ("r_specific", "#d62728")]:
        sub = at_deep.groupby("P")[comp].mean()
        axes[1].plot(sub.index, sub.values, marker="o", color=c, label=comp)
    axes[1].set_xscale("log"); axes[1].set_xlabel("P (number of perturbations)")
    axes[1].set_ylabel(f"reproducibility at n={n_deep}"); axes[1].legend(fontsize=8)
    axes[1].set_title("the P artifact: specific -> total as P falls")
    fig.suptitle("v5 G0 — Replogle K562 essential (guides pooled to gene level)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUT, "v5g0_replogle_k562_v1.png"), dpi=200, bbox_inches="tight")
    log("figures written")


if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "extract"
    if mode == "extract":
        extract()
    elif mode == "analyze":
        analyze()
    else:
        raise SystemExit("usage: v5g0_replogle_k562_kaggle_v1.py [extract|analyze]")